
extract_screenshots.py
----------------------
Reads a transcript .txt file, finds all time markers from a given
start time onwards, extracts a screenshot from the MP4 at each marker,
skips duplicate frames (pixel diff), and writes a matching .txt file
with the transcript segment(s) that correspond to each saved screenshot.
 
Requirements:
    pip install Pillow numpy
    ffmpeg in PATH  (https://ffmpeg.org/download.html)
 
Usage:
    1. Set VIDEO_PATH, TRANSCRIPT_PATH, OUTPUT_DIR, and START_TIME below.
    2. Run:  python extract_screenshots.py


In [ ]:

import re
import subprocess
import os
import tempfile
import base64
import json
import numpy as np
from PIL import Image
from dataclasses import dataclass
import anthropic

In [ ]:
# ─── CONFIGURATION ────────────────────────────────────────────────────────────
 
VIDEO_PATH           = "review_crm_meeting_recording.mp4"   # path to your MP4
TRANSCRIPT_PATH      = "transcript.txt"                     # path to your transcript
OUTPUT_DIR           = "screenshots"                        # output folder
START_TIME           = "0:03"                              # start here
END_TIME             = "59:53"                             # end here - None to process till the end
ANTHROPIC_API_KEY_PATH = "C:/code/apikeys/api_anthropic.key"     # path to your key file
 
# Mean pixel difference (0–255) below which two frames are considered identical.
# 5 is tight — right for screen recordings. Raise to 10–15 for camera video.
DIFF_THRESHOLD = 5
 
# ──────────────────────────────────────────────────────────────────────────────

The value is the **mean absolute pixel difference per pixel**, on a 0–255 scale (grayscale). Here's a practical breakdown:

| Threshold | Sensitivity | Skips when… |
|-----------|-------------|-------------|
| `1–3` | Extremely tight | Frames are virtually byte-identical. Even minor video compression artifacts may fool it into thinking frames differ. |
| `5` *(default)* | Tight | Safe for screen recordings — only saves when something visually meaningful changed on screen. |
| `10–15` | Moderate | Good for mixed content — screen share + occasional camera view of a person. Small movements (blinking, shifting) still get skipped. |
| `20–30` | Loose | Only skips frames that are nearly identical. A person moving in frame would likely pass through. |
| `50+` | Very loose | Only catches frames that are overwhelmingly the same — not very useful. |

**For your specific case** (screen recording of documents/forms), `5` is the right call. The transitions are hard cuts — one form disappears, another appears — so the diff jumps dramatically. There's no in-between. You could honestly go as low as `2` and it would still work perfectly.

Where it gets tricky is if the recording ever switches to a **camera view of the participants** — faces, subtle movements, and lighting changes produce diffs of `10–30` even on a "static" shot. In that scenario you'd want to raise the threshold to maybe `15–20` to avoid saving near-identical talking-head frames. But for pure screen share content, `5` is solid.

In [ ]:
FORM_ANALYSIS_PROMPT = """Analyze this screenshot and return ONLY a valid JSON object — no explanation, no markdown, no backticks.
 
If the screenshot contains a form, document, or structured data entry screen, extract its structure using this schema:
 
{
  "is_form": true,
  "title": "form title if visible, or null",
  "sections": [
    {
      "title": "section heading if visible, or null",
      "fields": [
        {
          "label": "field label",
          "type": "text|textarea|checkbox|radio|dropdown|date|number|signature|table|label_only|other",
          "options": ["option1", "option2"],
          "value": "pre-filled value if any, or null",
          "required": null
        }
      ]
    }
  ],
  "notes": "any additional observations about the form, or null"
}
 
If the screenshot does NOT contain a form (e.g. it is a conversation, desktop, browser, or blank screen), return:
 
{
  "is_form": false,
  "description": "brief description of what is visible"
}
 
Return ONLY the JSON object. Nothing else."""

In [ ]:
@dataclass
class Segment:
    seconds:  float
    time_str: str
    speaker:  str
    text:     str

In [34]:
def time_to_seconds(time_str: str) -> float:
    """M:SS | MM:SS | H:MM:SS  →  float seconds"""
    parts = [int(p) for p in time_str.strip().split(":")]
    if len(parts) == 2:
        return parts[0] * 60 + parts[1]
    if len(parts) == 3:
        return parts[0] * 3600 + parts[1] * 60 + parts[2]
    raise ValueError(f"Unrecognised time format: {time_str!r}")
 
 
def safe_name(time_str: str) -> str:
    """
    Convert a timestamp to a zero-padded HH-MM-SS filename-safe string.
    Ensures correct alphabetical/lexicographic sort in Windows Explorer.
      9:30     ->  00-09-30
      59:53    ->  00-59-53
      1:04:22  ->  01-04-22
    """
    parts = [int(p) for p in time_str.strip().split(":")]
    if len(parts) == 2:
        h, m, s = 0, parts[0], parts[1]
    else:
        h, m, s = parts[0], parts[1], parts[2]
    return f"{h:02d}-{m:02d}-{s:02d}"
 
 
def parse_transcript(transcript_path: str, start_time: str, end_time: str | None = None) -> list[Segment]:
    """
    Parse the transcript into Segment objects.
    Only returns segments between start_time and end_time (inclusive).
    Speaker lines look like:   "Tammy Salewski   1:04:22"
    """
    start_secs   = time_to_seconds(start_time)
    end_secs     = time_to_seconds(end_time) if end_time else float("inf")
    speaker_line = re.compile(r"^(.+?)\s{2,}(\d{1,2}:\d{2}(?::\d{2})?)\s*$")
 
    with open(transcript_path, "r", encoding="utf-8") as fh:
        lines = fh.readlines()
 
    segments: list[Segment] = []
    current_speaker = None
    current_time    = None
    current_secs    = 0.0
    buffer: list[str] = []
 
    def flush():
        if current_speaker and current_time and buffer:
            text = "\n".join(buffer).strip()
            if text and start_secs <= current_secs <= end_secs:
                segments.append(Segment(
                    seconds  = current_secs,
                    time_str = current_time,
                    speaker  = current_speaker,
                    text     = text,
                ))
 
    for line in lines:
        m = speaker_line.match(line.rstrip("\n"))
        if m:
            flush()
            buffer          = []
            current_speaker = m.group(1).strip()
            current_time    = m.group(2).strip()
            current_secs    = time_to_seconds(current_time)
        else:
            stripped = line.rstrip("\n")
            if stripped or buffer:
                buffer.append(stripped)
 
    flush()
    return segments
 
 
def extract_timestamps(transcript_path: str, start_time: str, end_time: str | None = None) -> list[str]:
    """Ordered, deduplicated list of time markers >= start_time (and <= end_time if set)."""
    start_secs = time_to_seconds(start_time)
    end_secs   = time_to_seconds(end_time) if end_time else float("inf")
    pattern    = re.compile(r"^.+\s+(\d{1,2}:\d{2}(?::\d{2})?)\s*$", re.MULTILINE)
 
    with open(transcript_path, "r", encoding="utf-8") as fh:
        content = fh.read()
 
    seen, result = set(), []
    for m in pattern.finditer(content):
        ts = m.group(1)
        if ts not in seen and start_secs <= time_to_seconds(ts) <= end_secs:
            seen.add(ts)
            result.append(ts)
    return result
 
 
def images_are_same(img_a: np.ndarray, img_b: np.ndarray) -> bool:
    """Mean absolute pixel diff on a 128x128 grayscale thumbnail."""
    size = (128, 128)
    a    = Image.fromarray(img_a).resize(size).convert("L")
    b    = Image.fromarray(img_b).resize(size).convert("L")
    diff = np.mean(np.abs(np.array(a, dtype=float) - np.array(b, dtype=float)))
    return diff < DIFF_THRESHOLD
 
 
def screenshot_at(video_path: str, time_str: str, output_dir: str,
                  prev_image: np.ndarray | None) -> tuple[str | None, np.ndarray | None]:
    """
    Extract frame at time_str, compare to prev_image.
    Returns (saved_path, image_array) if new, or (None, prev_image) if duplicate.
    """
    seconds   = time_to_seconds(time_str)
    fname = safe_name(time_str)
    out_path  = os.path.join(output_dir, f"screenshot_{fname}.jpg")
 
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
        tmp_path = tmp.name
 
    cmd = ["ffmpeg", "-ss", str(seconds), "-i", video_path,
           "-vframes", "1", "-q:v", "2", tmp_path, "-y"]
 
    result = subprocess.run(cmd, capture_output=True, text=True)
 
    if result.returncode != 0:
        print(f"  x  {time_str:>10}  FAILED")
        print(f"     {result.stderr.strip()[-300:]}")
        os.unlink(tmp_path)
        return None, prev_image
 
    current_image = np.array(Image.open(tmp_path))
 
    if prev_image is not None and images_are_same(prev_image, current_image):
        print(f"  ~  {time_str:>10}  skipped (duplicate)")
        os.unlink(tmp_path)
        return None, prev_image
 
    os.replace(tmp_path, out_path)
    print(f"  +  {time_str:>10}  ->  {out_path}")
    return out_path, current_image
 
 
def write_transcript_txt(segments: list[Segment],
                         saved_timestamps: list[str],
                         output_dir: str) -> None:
    """
    For each saved screenshot, write a matching .txt with all transcript
    segments whose time falls in [this_saved_time, next_saved_time).
    """
    if not saved_timestamps:
        return
 
    saved_secs = [time_to_seconds(ts) for ts in saved_timestamps]
 
    for i, ts in enumerate(saved_timestamps):
        window_start = saved_secs[i]
        window_end   = saved_secs[i + 1] if i + 1 < len(saved_secs) else float("inf")
 
        window_segments = [
            s for s in segments
            if window_start <= s.seconds < window_end
        ]
 
        fname     = safe_name(ts)
        txt_path  = os.path.join(output_dir, f"screenshot_{fname}.txt")
 
        with open(txt_path, "w", encoding="utf-8") as fh:
            if window_segments:
                blocks = []
                for seg in window_segments:
                    blocks.append(f"{seg.speaker}   {seg.time_str}\n{seg.text}")
                fh.write("\n\n".join(blocks) + "\n")
            else:
                fh.write("(no transcript content for this segment)\n")
 
        print(f"  doc  {ts:>10}  ->  {txt_path}  ({len(window_segments)} segment(s))")
 
 
def analyze_form_image(client: anthropic.Anthropic,
                       image_path: str,
                       output_dir: str,
                       time_str: str) -> None:
    """
    Send the screenshot to Claude vision, parse the JSON response,
    and save it as a matching .json file.
    The matching transcript .txt (if it exists) is prepended as context
    so the model understands what was being discussed on screen.
    """
    fname = safe_name(time_str)
    json_path = os.path.join(output_dir, f"screenshot_{fname}.json")
    txt_path  = os.path.join(output_dir, f"screenshot_{fname}.txt")
 
    # Read matching transcript context if available
    transcript_context = ""
    if os.path.isfile(txt_path):
        with open(txt_path, "r", encoding="utf-8") as f:
            txt_content = f.read().strip()
        if txt_content and txt_content != "(no transcript content for this segment)":
            transcript_context = (
                "The following is the transcript of the conversation that was happening "
                "while this screen was visible. Use it as context to better understand "
                "and label what you see in the screenshot:\n\n"
                f"{txt_content}\n\n"
            )
 
    # Base64-encode the image
    with open(image_path, "rb") as f:
        image_data = base64.standard_b64encode(f.read()).decode("utf-8")
 
    # Build the full prompt — transcript context first, then the analysis instructions
    full_prompt = transcript_context + FORM_ANALYSIS_PROMPT
 
    try:
        response = client.messages.create(
            model      = "claude-sonnet-4-20250514",
            max_tokens = 2048,
            messages   = [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": full_prompt,
                        },
                        {
                            "type": "image",
                            "source": {
                                "type"       : "base64",
                                "media_type" : "image/jpeg",
                                "data"       : image_data,
                            },
                        },
                    ],
                }
            ],
        )
 
        raw = response.content[0].text.strip()
 
        # Strip accidental markdown fences if the model adds them
        if raw.startswith("```"):
            raw = re.sub(r"^```[a-z]*\n?", "", raw)
            raw = re.sub(r"\n?```$", "", raw)
 
        parsed = json.loads(raw)
 
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, indent=2, ensure_ascii=False)
 
        is_form = parsed.get("is_form", False)
        tag     = "form" if is_form else "----"
        print(f"  {tag}  {time_str:>10}  ->  {json_path}")
 
    except json.JSONDecodeError as e:
        print(f"  !json  {time_str:>10}  JSON parse error: {e}")
        # Save the raw response anyway so nothing is lost
        with open(json_path, "w", encoding="utf-8") as f:
            f.write(raw)
    except Exception as e:
        print(f"  !err   {time_str:>10}  {e}")
 
 
def write_consolidated(saved_timestamps: list[str], output_dir: str) -> None:
    """
    Produce a single consolidated file containing all entries in order.
    Each entry is structured as:
 
        ══ [timestamp] ══════════════════════
        --- TRANSCRIPT ---
        <content of .txt>
 
        --- FORM ANALYSIS ---
        <content of .json>
    """
    out_path = os.path.join(output_dir, "consolidated.txt")
    written  = 0
 
    with open(out_path, "w", encoding="utf-8") as out:
        for ts in saved_timestamps:
            fname = safe_name(ts)
            txt_path  = os.path.join(output_dir, f"screenshot_{fname}.txt")
            json_path = os.path.join(output_dir, f"screenshot_{fname}.json")
 
            out.write(f"{'=' * 60}\n")
            out.write(f"  {ts}\n")
            out.write(f"{'=' * 60}\n\n")
 
            # --- transcript block ---
            out.write("--- TRANSCRIPT ---\n")
            if os.path.isfile(txt_path):
                with open(txt_path, "r", encoding="utf-8") as f:
                    out.write(f.read().strip())
            else:
                out.write("(no transcript file found)")
            out.write("\n\n")
 
            # --- form analysis block ---
            out.write("--- FORM ANALYSIS ---\n")
            if os.path.isfile(json_path):
                with open(json_path, "r", encoding="utf-8") as f:
                    raw = f.read().strip()
                # Pretty-print if valid JSON, otherwise write raw
                try:
                    out.write(json.dumps(json.loads(raw), indent=2, ensure_ascii=False))
                except json.JSONDecodeError:
                    out.write(raw)
            else:
                out.write("(no json file found)")
            out.write("\n\n")
 
            written += 1
 
    print(f"\nConsolidated file written: {out_path}  ({written} entries)")

In [29]:
def load_saved_timestamps(output_dir: str) -> list[str]:
    """
    Reconstruct saved_timestamps from jpg files already in output_dir.
    Filenames like screenshot_59-53.jpg  →  59:53
                 screenshot_1-00-14.jpg  →  1:00:14
    """
    pattern = re.compile(r"^screenshot_(\d{1,2}-\d{2}(?:-\d{2})?)\.jpg$")
    timestamps = []
    for fname in os.listdir(output_dir):
        m = pattern.match(fname)
        if m:
            ts = m.group(1).replace("-", ":", 2)  # restore colons
            timestamps.append(ts)
    return sorted(timestamps, key=time_to_seconds)

In [30]:
# ── sanity checks ──────────────────────────────────────────────────────────
if not os.path.isfile(VIDEO_PATH):
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH!r}")
if not os.path.isfile(TRANSCRIPT_PATH):
    raise FileNotFoundError(f"Transcript not found: {TRANSCRIPT_PATH!r}")
if not os.path.isfile(ANTHROPIC_API_KEY_PATH):
    raise FileNotFoundError(f"API key file not found: {ANTHROPIC_API_KEY_PATH!r}")

with open(ANTHROPIC_API_KEY_PATH, "r") as f:
    api_key = f.read().strip()

claude = anthropic.Anthropic(api_key=api_key)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [31]:
# ── parse transcript ───────────────────────────────────────────────────────
print(f"\nParsing transcript from {START_TIME} onwards ...")
segments   = parse_transcript(TRANSCRIPT_PATH, START_TIME, END_TIME)
timestamps = extract_timestamps(TRANSCRIPT_PATH, START_TIME, END_TIME)

if not timestamps:
    print("No time markers found. Check your transcript format.")
else:
    print(f"Found {len(timestamps)} timestamp(s), {len(segments)} transcript segment(s)\n")


Parsing transcript from 0:03 onwards ...
Found 130 timestamp(s), 135 transcript segment(s)



In [32]:
# ── extract screenshots ────────────────────────────────────────────────────
print(f"Extracting screenshots ...\n")
prev_image       = None
saved            = 0
skipped          = 0
saved_timestamps: list[str] = []
saved_paths: dict[str, str] = {}   # ts -> jpg path

for ts in timestamps:
    path, prev_image = screenshot_at(VIDEO_PATH, ts, OUTPUT_DIR, prev_image)
    if path:
        saved += 1
        saved_timestamps.append(ts)
        saved_paths[ts] = path
    else:
        skipped += 1

print(f"\n-- {saved} saved, {skipped} duplicates skipped (of {len(timestamps)} total)\n")

Extracting screenshots ...

  +        0:03  ->  screenshots\screenshot_00-00-03.jpg
  +        0:09  ->  screenshots\screenshot_00-00-09.jpg
  ~        0:13  skipped (duplicate)
  +        0:56  ->  screenshots\screenshot_00-00-56.jpg
  ~        1:00  skipped (duplicate)
  ~        1:04  skipped (duplicate)
  +        1:49  ->  screenshots\screenshot_00-01-49.jpg
  +        5:48  ->  screenshots\screenshot_00-05-48.jpg
  +        5:50  ->  screenshots\screenshot_00-05-50.jpg
  ~        6:45  skipped (duplicate)
  ~        7:25  skipped (duplicate)
  ~        7:44  skipped (duplicate)
  ~        7:49  skipped (duplicate)
  ~        8:10  skipped (duplicate)
  ~        8:24  skipped (duplicate)
  +        9:15  ->  screenshots\screenshot_00-09-15.jpg
  +        9:30  ->  screenshots\screenshot_00-09-30.jpg
  +       11:13  ->  screenshots\screenshot_00-11-13.jpg
  ~       11:26  skipped (duplicate)
  +       11:58  ->  screenshots\screenshot_00-11-58.jpg
  +       12:20  ->  screenshots

In [35]:
# ── write transcript txt files ─────────────────────────────────────────────
print("Writing transcript txt files ...\n")
write_transcript_txt(segments, saved_timestamps, OUTPUT_DIR)

Writing transcript txt files ...

  doc        0:03  ->  screenshots\screenshot_00-00-03.txt  (2 segment(s))
  doc        0:09  ->  screenshots\screenshot_00-00-09.txt  (3 segment(s))
  doc        0:56  ->  screenshots\screenshot_00-00-56.txt  (3 segment(s))
  doc        1:49  ->  screenshots\screenshot_00-01-49.txt  (1 segment(s))
  doc        5:48  ->  screenshots\screenshot_00-05-48.txt  (1 segment(s))
  doc        5:50  ->  screenshots\screenshot_00-05-50.txt  (7 segment(s))
  doc        9:15  ->  screenshots\screenshot_00-09-15.txt  (1 segment(s))
  doc        9:30  ->  screenshots\screenshot_00-09-30.txt  (1 segment(s))
  doc       11:13  ->  screenshots\screenshot_00-11-13.txt  (2 segment(s))
  doc       11:58  ->  screenshots\screenshot_00-11-58.txt  (1 segment(s))
  doc       12:20  ->  screenshots\screenshot_00-12-20.txt  (1 segment(s))
  doc       13:02  ->  screenshots\screenshot_00-13-02.txt  (1 segment(s))
  doc       13:16  ->  screenshots\screenshot_00-13-16.txt  (1 seg

In [36]:
# ── analyze screenshots with Claude vision ─────────────────────────────────
print(f"\nAnalyzing screenshots with Claude vision ...\n")
for ts in saved_timestamps:
    analyze_form_image(claude, saved_paths[ts], OUTPUT_DIR, ts)
    

print(f"\nDone -- {saved} screenshot/txt/json triplet(s) in '{OUTPUT_DIR}/'")


Analyzing screenshots with Claude vision ...

  ----        0:03  ->  screenshots\screenshot_00-00-03.json
  ----        0:09  ->  screenshots\screenshot_00-00-09.json
  ----        0:56  ->  screenshots\screenshot_00-00-56.json
  ----        1:49  ->  screenshots\screenshot_00-01-49.json
  form        5:48  ->  screenshots\screenshot_00-05-48.json
  form        5:50  ->  screenshots\screenshot_00-05-50.json
  ----        9:15  ->  screenshots\screenshot_00-09-15.json
  ----        9:30  ->  screenshots\screenshot_00-09-30.json
  ----       11:13  ->  screenshots\screenshot_00-11-13.json
  ----       11:58  ->  screenshots\screenshot_00-11-58.json
  form       12:20  ->  screenshots\screenshot_00-12-20.json
  ----       13:02  ->  screenshots\screenshot_00-13-02.json
  ----       13:16  ->  screenshots\screenshot_00-13-16.json
  ----       13:31  ->  screenshots\screenshot_00-13-31.json
  form       14:15  ->  screenshots\screenshot_00-14-15.json
  ----       15:49  ->  screenshots\sc

In [37]:
# ── write consolidated file ───────────────────────────────────────────────

# Comment out the pipeline above and uncomment this to skip re-processing:
saved_timestamps = load_saved_timestamps(OUTPUT_DIR)

write_consolidated(saved_timestamps, OUTPUT_DIR)

print(f"\nDone -- {saved} screenshot/txt/json triplet(s) in '{OUTPUT_DIR}/'")


Consolidated file written: screenshots\consolidated.txt  (105 entries)

Done -- 39 screenshot/txt/json triplet(s) in 'screenshots/'
